# 04.4 — Public data and authentication

[Class index and setup](README.md) · [Previous: 04.3](04.3-failures-and-ai-api-previews.ipynb) · [Next: 04.5](04.5-testing-and-consolidation.ipynb)

**Purpose:** Apply the same small client to real public data without changing the report model.

**In class:** 0 minutes. **Independent work:** 60 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's separate project is additional work.

## Learning outcomes

- Read the minimum API documentation needed for a specific request.
- Map a World Bank response into stable internal observations.
- Explain query parameters versus a JSON body versus an authentication header.

## Starting point

You know basic Python, functions, JSON-like dictionaries, and the earlier Observation/Report idea. The complete package and fixtures are supplied again; no previous notebook or exercise answer is needed. Open the supplied class workspace in VS Code, select this class's `.venv`, then **Restart Kernel → Run All**.

Default cells inspect saved responses and run offline checks. They do not start a server, call an external API, read credentials, install software, or download models. Live commands are explicit **manual terminal steps**. An HTTP-shaped fixture is not evidence of a live request. Empty answer cells are safe.

Attempt exercises before using the separate `%load` cell: uncomment its last line and run to reveal, then run the populated cell separately. A loaded solution replaces its loader cell; restarting the kernel does not restore that source. Keep your attempt in its own cell.

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/api.py").is_file() or not Path("fixtures/observations.json").is_file():
    raise FileNotFoundError(
        "Open 04-consuming-apis.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available; default execution is offline.")

## Independent work · 60 minutes — A public-data adapter

Allocate **20 minutes** to documentation and the guided examples, then **40 minutes** to Exercise 04.4.H1, including its checks and explanation. There is no additional setup assignment.

The World Bank Indicators API needs no API key. Our request asks only for **Portugal and Spain**, indicator **SP.POP.TOTL** (population), and **2023**. A two-country, one-year query fits on one page. Start with the saved response, then explicitly try the live endpoint if network access permits.

| Request component | Meaning |
| --- | --- |
| `/v2/country/PRT;ESP/indicator/SP.POP.TOTL` | Two countries and one indicator |
| `format=json` | Ask for JSON rather than the default format |
| `date=2023` | Keep observations in one year |
| `per_page=2` | Enough records for this fixed query |

The response is a two-element list: metadata followed by records. That external shape is different from our local service's list of observations. It belongs at an adapter boundary, not in the report calculation.

Optional illustration: [API-in-context diagram and credit](IMAGE_SUGGESTIONS.md). Locate the client/service interface, then identify the adapter that maps external JSON into our unchanged internal record.

In [ ]:
from report_app.source import read_fixture
from report_app.public_data import POPULATION_URL, POPULATION_PARAMS, parse_population
from report_app.analysis import summarize

payload = read_fixture("world-bank-population.json")
print("URL:", POPULATION_URL)
print("Parameters:", POPULATION_PARAMS)
print("Saved response metadata:", payload[0])
print("First external record:", payload[1][0])
records = parse_population(payload)
print("Internal records:", [record.model_dump() for record in records])
print("Two-country summary:", summarize(records))

### Make the mapping explicit

| External field | Internal meaning |
| --- | --- |
| `countryiso3code` + `date` | `label`, for example `PRT:2023` |
| `value` | Population in **persons**, stored as `Observation.value` |
| Metadata `pages` | Must be one for this deliberately bounded example |

The `PopulationRow` model checks the source fields; the adapter then creates our existing `Observation`. A missing/null population must fail, not become zero. We also reject a multi-page envelope rather than silently reporting its first page as complete. You do not need to implement pagination for the required task.

The summary total is **the sum for these two selected countries in this year**. It is not world population. A lockfile fixes Python dependencies, not the values a public API will return after statistical revisions. The saved fixture records the authoring snapshot; live answers can differ. See [fixture provenance](fixtures/README.md) for source, retrieval, and attribution.

In [ ]:
import inspect
from report_app.public_data import parse_population

print(inspect.getsource(parse_population))

### Explicit live request — manual terminal step

From this class directory:

```sh
uv run --locked python -m report_app --public
```

This command contacts the World Bank, uses the same five-second Requests timeout and status/JSON checks, maps the response, and calculates the summary. The FastAPI server is not needed for this command. Run once, inspect the result, and keep testing with saved data. Do not repeatedly call a public service inside a test loop.

**Offline recovery:** if the service is unavailable, show the saved mapping above and record “saved-response integration; live request unavailable.” This demonstrates code and deterministic behavior, not current remote availability. Do not invent a live result.

### Three places to put request information

- A **query parameter** selects data in the URL: our GET uses `params={...}`.
- A **JSON body** supplies structured request content: the AI POST examples use `json={...}`.
- A **header** supplies request metadata: the OpenAI example uses `Authorization: Bearer ...`.

These locations are part of each service's contract; they are not interchangeable. A World Bank call needs no key. The local Ollama example assumes a local server and an installed model. The OpenAI demo reads credentials only in its explicit live path. Do not add fake authentication to the public API simply to practice headers.

### 🧩 Exercise 04.4.H1 — Integrate one public endpoint

> **Independent work · 40 minutes, included in this part’s 60**
>
> **Goal:** Adapt an external schema while preserving the internal data contract.
>
> **Start here:** The saved World Bank response, POPULATION_URL, POPULATION_PARAMS, and public_data.py.
>
> **Task:** Inspect the metadata and record fields; document the mapping to label/value and the unit. Run the saved adapter and attempt the explicit live command once if available. In a scratch payload set one value to None and explain the rejection. Record saved and live evidence separately.
>
> **Check:** You obtain PRT:2023 and ESP:2023 observations, identify persons as the unit, reject missing values, and explain why the result is only a two-country total. State whether your live attempt succeeded.
>
> **Hint:** Read payload[0] as metadata and payload[1] as records. Missing population is not zero. Keep HTTP outside summarize.

In [ ]:
# Your attempt for 04.4.H1.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_4_h1.py

## Completion and reflection — included in the exercise

Explain why the local and World Bank responses can feed the same summary despite their different JSON structures. Name the only part that changes when the service's envelope changes. Explain why a saved fixture is reliable for tests but does not establish the current population estimate or service uptime.

## References

- [World Bank API overview and key-free access](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation)
- [World Bank request parameters](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [Population indicator](https://data.worldbank.org/indicator/SP.POP.TOTL)
- [World Bank data terms](https://data.worldbank.org/summary-terms-of-use)
- [AI examples and instructor configuration](AI_DEMOS.md)